# Week 10 · Topic 5 — Splitting Code into a Package

**Box we are building: STORE, organised**

One long file is fine for a first program. By Week 10 your project has users, data, a database and (soon) web pages.
This topic shows how to give each job its own file, and how Python finds them.

> **Already know this?** Skip to the *Topic 5 Challenge*.

> **Setup.** The next cell builds a small practice package in a folder called `practice_pkg` next to this notebook, so you can import from it.
> It is safe to run more than once.

In [ ]:
import sys
from pathlib import Path

root = Path("practice_pkg")
(root / "shop").mkdir(parents=True, exist_ok=True)

(root / "shop" / "__init__.py").write_text("")          # an EMPTY file: it marks the folder as a package
(root / "shop" / "prices.py").write_text(
"""def add_vat(price, rate=0.2):
    return round(price * (1 + rate), 2)


def apply_discount(price, percent):
    return round(price * (1 - percent / 100), 2)
""")
(root / "shop" / "labels.py").write_text(
"""def make_label(name, price):
    return f"{name}: {price} GBP"
""")

if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))             # in a notebook we tell Python where to look
print(sorted(p.name for p in (root / "shop").iterdir()))

---
## 1. A module is just a `.py` file

Any Python file can be **imported** by another one. The file `prices.py` is a *module*, and its name (without `.py`) is what you import:

```python
from shop.prices import add_vat      # from <folder>.<file> import <function>
```

In [ ]:
from shop.prices import add_vat, apply_discount

print(add_vat(10))
print(apply_discount(10, 25))

### Your turn
Import `make_label` from `shop.labels` and print a label for a product of your choice, using `add_vat` to work out its price.

In [ ]:
# your code here

---
## 2. A package is a folder with an `__init__.py`

`shop` is a **package**: a folder that holds modules and a file called `__init__.py`. That file can be completely empty. It tells Python
"this folder is importable".

```
shop/
  __init__.py       <- empty, but it must be there
  prices.py
  labels.py
```

Your project does exactly this with `app_model/`:

```
app_model/
  __init__.py
  db.py             from app_model.db import connect_database
  users.py          from app_model.users import register_user
```

Ways to import (all do the same job):

| You write | You then use |
|---|---|
| `from shop.prices import add_vat` | `add_vat(10)` |
| `import shop.prices` | `shop.prices.add_vat(10)` |
| `from shop.prices import add_vat as vat` | `vat(10)` |

Prefer the first one: it is short and shows exactly where each function comes from.

### Your turn
Add a third function to `practice_pkg/shop/prices.py` (open the file in VS Code), for example `total(prices)` that adds a list of prices.
Then import and call it. (A notebook remembers old versions: run `import importlib, shop.prices; importlib.reload(shop.prices)` first.)

In [ ]:
# your code here

---
## 3. Where does Python look?

When you run `python main.py`, Python looks for the packages you import **in the folder that holds `main.py`**.
That is why your project keeps `main.py`, `load_data.py` and the `app_model` folder side by side, in `CST1510_CW2`.

If Python cannot find the package you get:

```
ModuleNotFoundError: No module named 'app_model'
```

The usual causes:

| Cause | Fix |
|---|---|
| the script is in a different folder from `app_model/` | keep the script next to the package folder |
| `app_model` is spelled differently in the import | the import must match the folder name exactly, including capitals |
| you ran the command from the wrong place (pytest, notebooks) | `cd` into `CST1510_CW2` first |

---
## 4. One job per file

The reason to split is that each file does one thing and you can find it again:

| File | Its one job |
|---|---|
| `db.py` | connect to the database |
| `schema.py` | create the tables |
| `users.py` | everything about users |
| `records.py` | everything about your data |

Next to this notebook, `examples/01_a_tiny_package/` is a complete small version. Open `run.py`: it makes **one connection** in one place and
passes it to every function that needs it. The functions never connect themselves:

```python
conn = connect_database()
create_pet_table(conn)
add_pet(conn, "Rex", 4)
```

That is what `conn` as the first parameter means in your project: *"here is the database to use"*.

### Your turn
Run `examples/01_a_tiny_package/run.py` (from inside that folder). Then add a `count_pets(conn)` function to `pets.py` and print its result from `run.py`.

---
## Topic 5 Challenge

Make your own package called `mytools` with two modules: `text.py` (a function `shout(word)` that returns the word in capitals with `!` after it) and
`maths.py` (a function `average(numbers)`). Add the `__init__.py`. Then, from a new `.py` file next to the `mytools` folder, import both functions and
print one result from each.

In [ ]:
# your code here

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | `loading.py`, `cleaning.py`, `models.py` instead of one 800-line notebook |
| **Cyber Security** | `logs.py` reads logs, `rules.py` decides what is suspicious, `report.py` prints the findings |
| **IT** | `inventory.py`, `tickets.py` and `alerts.py` sharing one database connection |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| no `__init__.py` | imports fail or behave oddly | create an empty `__init__.py` in the folder |
| running the script from the wrong folder | `ModuleNotFoundError: No module named 'app_model'` | keep the script next to the package; `cd` into the project folder |
| name of the file and the import do not match | `ImportError: cannot import name ...` | check the spelling of the file name and the function name |
| two files that import each other | `ImportError ... partially initialized module` | move the shared function into a third file |
| editing a module while a notebook is open | the old version still runs | `importlib.reload(module)` or restart the kernel |

## Go deeper

`examples/01_a_tiny_package/` — a three-file package used by one script
`examples/02_import_errors.py` — what `ModuleNotFoundError` looks like and why
Docs: [Python modules and packages](https://docs.python.org/3/tutorial/modules.html)

---
## Recap

| You can now | Using |
|---|---|
| import a function from your own file | `from folder.file import function` |
| make a folder importable | an empty `__init__.py` |
| understand `ModuleNotFoundError` | Python looks in the folder of the script you run |
| share one database connection | pass `conn` as the first parameter |

**Next:** back to your lab. Step 5 builds `app_model/` exactly like this.